In [5]:
%pip install pandas
import pandas as pd

Defaulting to user installation because normal site-packages is not writeable
     |████████████████████████████████| 10.8 MB 4.9 MB/s eta 0:00:01
     |████████████████████████████████| 348 kB 79.6 MB/s eta 0:00:01
     |████████████████████████████████| 5.3 MB 92.3 MB/s eta 0:00:01
     |████████████████████████████████| 510 kB 77.6 MB/s eta 0:00:01
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
stats = pd.read_csv('data/advanced_stats_all.csv')

In [ ]:
stats_clean = stats.copy()

# Remove repeated header rows BBRef injects mid-table
stats_clean = stats_clean[stats_clean['Player'] != 'Player']
stats_clean = stats_clean[stats_clean['Player'].notna()]

# Drop columns not useful for modeling
stats_clean = stats_clean.drop(columns=['Rk', 'Player-additional', 'Awards'], errors='ignore')

# Convert numeric columns (they come in as strings/objects from BBRef scrape)
numeric_cols = ['Age', 'G', 'GS', 'MP', 'PER', 'TS%', '3PAr', 'FTr',
                'ORB%', 'DRB%', 'TRB%', 'AST%', 'STL%', 'BLK%', 'TOV%', 'USG%',
                'OWS', 'DWS', 'WS', 'WS/48', 'OBPM', 'DBPM', 'BPM', 'VORP']
for col in numeric_cols:
    if col in stats_clean.columns:
        stats_clean[col] = pd.to_numeric(stats_clean[col], errors='coerce')

# Filter out players with very low minutes (small sample noise)
stats_clean = stats_clean[stats_clean['MP'] >= 500]

# For traded players, BBRef adds a 'TOT' row (totals) + individual team rows
# Keep only the TOT row so each player appears once per season
traded = stats_clean[stats_clean['Team'] == 'TOT']['Player'].unique()
stats_clean = stats_clean[
    ~((stats_clean['Player'].isin(traded)) & (stats_clean['Team'] != 'TOT'))
]

# Drop any remaining duplicates (same player, same season)
stats_clean = stats_clean.drop_duplicates(subset=['Player', 'Season'], keep='first')

# Clean up text columns
for col in ['Player', 'Team', 'Pos']:
    if col in stats_clean.columns:
        stats_clean[col] = stats_clean[col].astype(str).str.strip()

# Drop rows where key stats are all missing
stats_clean = stats_clean.dropna(subset=['PER', 'VORP', 'BPM', 'WS'])

stats_clean = stats_clean.reset_index(drop=True)

stats_clean.to_csv('data/advanced_stats_clean.csv', index=False)

print("Cleaned shape:", stats_clean.shape)
print("Seasons:", sorted(stats_clean['Season'].unique()))
print(stats_clean[['Player', 'Season', 'Season_Label', 'Age', 'Pos', 'MP', 'PER', 'VORP', 'BPM', 'WS']].head(10))